# 🧩 Dataset Consolidation: Combining Bounding Boxes and Images

## Overview
This notebook implements the final stage of the dataset preparation pipeline for medical handwriting text recognition. After extracting bounding boxes from multiple medical document images using YOLO detection, we need to consolidate all the scattered bounding box images and their corresponding labels into a unified, organized dataset structure.

## Purpose and Context
In the medical HTR workflow, each source document generates multiple bounding box images (one per detected text line) stored in separate subdirectories. This creates a fragmented dataset structure that needs to be unified before training the text recognition model. The consolidation process serves several critical purposes:

1. **Unified Dataset Structure**: Creates a single directory containing all bounding box images with sequential naming
2. **Label Consolidation**: Merges all individual label files into one comprehensive labels file
3. **Data Integrity**: Ensures proper mapping between images and their corresponding text labels
4. **Training Preparation**: Organizes data in the format required by TrOCR training pipeline

## Technical Implementation Details

### Input Structure
The process expects the following input directory structure:
```
extracted_bounding_boxes/
├── IMG_1368/
│   ├── line1.jpg, line2.jpg, ...
│   └── bounding_box_texts_IMG_1368.txt
├── IMG_1370/
│   ├── line1.jpg, line2.jpg, ...
│   └── bounding_box_texts_IMG_1370.txt
└── ... (additional image folders)
```

### Output Structure
The consolidation process generates:
```
combined_bboxes/
├── line1.jpg, line2.jpg, ..., lineN.jpg (sequentially numbered)
└── combined_labels.txt (unified labels file)
```

### Key Features
- **Sequential Renaming**: All bounding box images are renamed with sequential line numbers (line1.jpg, line2.jpg, etc.)
- **Label Mapping**: Updates label file to reflect new sequential naming while preserving original text content
- **Source Tracking**: Maintains traceability by logging which original image each line came from
- **File Integrity**: Ensures all images are successfully copied and labeled before proceeding

### Expected Results
After successful execution, you should have:
- A unified directory with thousands of sequentially-named bounding box images
- A single labels file mapping each image to its corresponding handwritten text
- Complete consolidation of the scattered dataset into training-ready format

This consolidation step is essential for the subsequent TrOCR fine-tuning process, as it expects a clean, organized dataset structure with consistent naming conventions.

In [ ]:
import os
import shutil

def setup_consolidation_directories():
    """
    Sets up the required directory structure for bounding box consolidation.
    
    Creates the output directory for combined bounding boxes and initializes
    the paths used throughout the consolidation process.
    
    Returns:
        tuple: A tuple containing (main_directory, output_directory, combined_labels_file)
    """
    # Path to directory containing extracted bounding boxes from individual images
    main_directory = "enter your path here"  # Path to extracted_bounding_boxes folder
    
    # Output directory for consolidated dataset
    output_directory = "enter your path here"  # Path for combined_bboxes output folder
    
    # Combined labels file name
    combined_labels_file = "enter your path here"  # Path for combined_labels.txt file
    
    # Create the output directory if it doesn't exist
    os.makedirs(output_directory, exist_ok=True)
    
    return main_directory, output_directory, combined_labels_file

def process_subfolder_labels(subfolder_path, subfolder_name):
    """
    Processes label files from a specific subfolder and extracts line information.
    
    Each subfolder contains a text file with line numbers and corresponding text labels.
    This function reads and parses these labels for processing.
    
    Args:
        subfolder_path (str): Path to the subfolder containing bounding box images
        subfolder_name (str): Name of the subfolder (typically image name like IMG_1368)
    
    Returns:
        list: List of tuples containing (original_line_number, label_text)
    """
    # Construct the label file name based on subfolder naming convention
    label_file_name = f"bounding_box_texts_{subfolder_name}.txt"
    label_file_path = os.path.join(subfolder_path, label_file_name)
    
    extracted_labels = []
    
    if os.path.exists(label_file_path):
        # Read the text file for labels in the current subfolder
        with open(label_file_path, "r", encoding='utf-8') as label_file:
            lines = label_file.readlines()

        # Process each line in the text file
        for line in lines:
            if ': ' in line.strip():  # Ensure line has expected format
                # Extract the original line number and label
                original_line_number, label = line.strip().split(': ', 1)
                
                # Clean up the line number (e.g., remove "Line " prefix)
                original_line_number = original_line_number.replace("Line ", "").strip()
                
                extracted_labels.append((original_line_number, label))
    
    return extracted_labels

def copy_and_rename_images(subfolder_path, subfolder_name, extracted_labels, 
                          current_line_number, output_directory, combined_labels):
    """
    Copies bounding box images from subfolder to output directory with sequential naming.
    
    This function handles the core consolidation logic by:
    1. Renaming images from original naming (line1.jpg, line2.jpg) to sequential global naming
    2. Copying renamed images to the consolidated output directory
    3. Updating the combined labels list with new sequential naming
    
    Args:
        subfolder_path (str): Path to source subfolder
        subfolder_name (str): Name of the subfolder for tracking purposes
        extracted_labels (list): List of (line_number, label) tuples
        current_line_number (int): Current global line number for sequential naming
        output_directory (str): Destination directory for consolidated images
        combined_labels (list): List to accumulate all labels with new naming
    
    Returns:
        int: Updated current_line_number after processing this subfolder
    """
    for original_line_number, label in extracted_labels:
        # Generate original and new image file names
        original_image_name = f"line{original_line_number}.jpg"
        new_image_name = f"line{current_line_number}.jpg"
        
        # Construct full paths
        original_image_path = os.path.join(subfolder_path, original_image_name)
        new_image_path = os.path.join(output_directory, new_image_name)
        
        # Copy the renamed image to the output directory
        if os.path.exists(original_image_path):
            shutil.copyfile(original_image_path, new_image_path)
            
            # Append the updated line to the combined labels list
            combined_labels.append(f"line{current_line_number}: {label}\\n")
            
            # Log progress for verification
            print(f"Processed: {current_line_number} | {original_image_name} | {subfolder_name}")
            
            # Increment the current line number
            current_line_number += 1
        else:
            print(f"Warning: Image {original_image_path} not found")
    
    return current_line_number

def consolidate_bounding_boxes():
    """
    Main function to consolidate all bounding box images and labels into a unified dataset.
    
    This function orchestrates the complete consolidation process:
    1. Sets up directory structure
    2. Iterates through all subfolders containing bounding box extractions
    3. Processes labels and copies images with sequential naming
    4. Creates a unified labels file
    
    The consolidation ensures that all bounding box images from different source documents
    are combined into a single, organized dataset ready for TrOCR training.
    
    Returns:
        tuple: (total_processed_images, output_directory, combined_labels_file)
    """
    # Initialize directory structure and paths
    main_directory, output_directory, combined_labels_file = setup_consolidation_directories()
    
    # Initialize consolidation tracking variables
    current_line_number = 1
    combined_labels = []
    
    print(f"Starting consolidation from: {main_directory}")
    print(f"Output directory: {output_directory}")
    print("-" * 50)
    
    # Iterate through each subfolder in the main directory
    for subfolder in sorted(os.listdir(main_directory)):
        subfolder_path = os.path.join(main_directory, subfolder)
        
        # Process only directories (skip files)
        if os.path.isdir(subfolder_path):
            print(f"Processing subfolder: {subfolder}")
            
            # Extract labels from the subfolder's text file
            extracted_labels = process_subfolder_labels(subfolder_path, subfolder)
            
            if extracted_labels:
                # Copy images and update labels with sequential naming
                current_line_number = copy_and_rename_images(
                    subfolder_path, subfolder, extracted_labels,
                    current_line_number, output_directory, combined_labels
                )
            else:
                print(f"No labels found for subfolder: {subfolder}")
    
    # Write the combined labels to a single text file
    with open(combined_labels_file, "w", encoding='utf-8') as combined_file:
        combined_file.writelines(combined_labels)
    
    total_processed = current_line_number - 1
    
    print("-" * 50)
    print(f"Consolidation complete!")
    print(f"Total images processed: {total_processed}")
    print(f"All bounding box images are in: {output_directory}")
    print(f"Combined labels saved in: {combined_labels_file}")
    
    return total_processed, output_directory, combined_labels_file

# Execute the consolidation process
if __name__ == "__main__":
    total_images, output_dir, labels_file = consolidate_bounding_boxes()